In [ ]:
# ── INSTALL ───────────────────────────────────────────────
!pip install roboflow opencv-python --quiet

from roboflow import Roboflow
from pathlib import Path
import cv2
import shutil

# ── CONFIG ───────────────────────────────────────────────
ROBOFLOW_API_KEY = "vLosZUv2drPENvUknWRP"
WORKSPACE = "samyaks-workspace"
PROJECT   = "indicraft"
VERSION   = 2

DOWNLOAD_DIR = Path("/content/drive/MyDrive/dlcv_phase_2/IndiCraft-Tools-Dataset")
OUT_ROOT     = Path("/content/drive/MyDrive/dlcv_phase_2/IndiCraft-crops")

PADDING = 0.05

CLASS_NAMES = [
    "wood_chisel", "anvil", "blacksmith_hammer", "blacksmith_tongs",
    "carpentry_hammer", "hand_planer", "hand_saw", "pottery_wheel"
]

# ── STEP 1: DOWNLOAD ─────────────────────────────────────
def download_dataset():
    if DOWNLOAD_DIR.exists():
        print(f"[INFO] Dataset already exists at {DOWNLOAD_DIR}")
        return DOWNLOAD_DIR

    print("[INFO] Downloading from Roboflow...")
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace(WORKSPACE).project(PROJECT)

    dataset = project.version(VERSION).download(
        "yolov9",
        location=str(DOWNLOAD_DIR)
    )

    print(f"[DONE] Downloaded to {dataset.location}")
    return Path(dataset.location)

# ── STEP 2: FIND SPLITS ───────────────────────────────────
def find_split_dirs(base):
    splits = {}

    for split in ["train", "valid", "test"]:
        img_a = base / split / "images"
        lbl_a = base / split / "labels"

        img_b = base / "images" / split
        lbl_b = base / "labels" / split

        if img_a.exists():
            splits[split] = (img_a, lbl_a)
        elif img_b.exists():
            splits[split] = (img_b, lbl_b)

    if "valid" in splits:
        splits["val"] = splits.pop("valid")

    return splits

# ── STEP 3: CLEAN OUTPUT (IMPORTANT) ──────────────────────
def reset_output():
    if OUT_ROOT.exists():
        print("[INFO] Clearing old crops...")
        shutil.rmtree(OUT_ROOT)
    OUT_ROOT.mkdir(parents=True, exist_ok=True)

# ── STEP 4: CROP ─────────────────────────────────────────
def crop_split(split, img_dir, lbl_dir):
    count = 0
    images = list(img_dir.glob("*.*"))

    for i, img_path in enumerate(images):
        if i % 200 == 0:
            print(f"[{split}] Processing {i}/{len(images)}")

        label_path = lbl_dir / (img_path.stem + ".txt")
        if not label_path.exists():
            continue

        img = cv2.imread(str(img_path))
        if img is None:
            continue

        H, W = img.shape[:2]

        with open(label_path) as f:
            lines = [l.strip() for l in f if l.strip()]

        for idx, line in enumerate(lines):
            cls_id, cx, cy, bw, bh = map(float, line.split())

            cls_id = int(cls_id)
            if cls_id >= len(CLASS_NAMES):
                continue

            # Convert YOLO → pixel coords
            pad_w = bw * PADDING * W
            pad_h = bh * PADDING * H

            x1 = int((cx - bw/2)*W - pad_w)
            y1 = int((cy - bh/2)*H - pad_h)
            x2 = int((cx + bw/2)*W + pad_w)
            y2 = int((cy + bh/2)*H + pad_h)

            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(W, x2), min(H, y2)

            if x2 <= x1 or y2 <= y1:
                continue

            crop = img[y1:y2, x1:x2]
            cls_name = CLASS_NAMES[cls_id]

            save_dir = OUT_ROOT / split / cls_name
            save_dir.mkdir(parents=True, exist_ok=True)

            out_name = f"{img_path.stem}_inst{idx}.jpg"
            cv2.imwrite(str(save_dir / out_name), crop)

            count += 1

    print(f"[DONE] {split}: {count} crops")

# ── STEP 5: VERIFY ───────────────────────────────────────
def verify():
    print("\n=== DATASET SUMMARY ===")
    total = 0

    for split in ["train", "val", "test"]:
        print(f"\n{split.upper()}:")
        for cls in CLASS_NAMES:
            d = OUT_ROOT / split / cls
            n = len(list(d.glob("*.jpg"))) if d.exists() else 0
            print(f"  {cls:<22} {n}")
            total += n

    print(f"\nTOTAL CROPS: {total}")
    print(f"Path: {OUT_ROOT}")

# ── MAIN ─────────────────────────────────────────────────
if __name__ == "__main__":

    dataset_path = download_dataset()
    splits = find_split_dirs(dataset_path)

    print("\nFound splits:", splits.keys())

    reset_output()

    for split, (img_dir, lbl_dir) in splits.items():
        crop_split(split, img_dir, lbl_dir)

    verify()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.0/184.0 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 86.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 114.3 MB/s eta 0:00:00
[INFO] Downloading from Roboflow...
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/drive/MyDrive/dlcv_phase_2/IndiCraft-Tools-Dataset in yolov9:: 100%|██████████| 4143/4143 [00:48<00:00, 85.01it/s]


[DONE] Downloaded to /content/drive/MyDrive/dlcv_phase_2/IndiCraft-Tools-Dataset

Found splits: dict_keys(['train', 'test', 'val'])
[train] Processing 0/1512
[train] Processing 200/1512
[train] Processing 400/1512
[train] Processing 600/1512
[train] Processing 800/1512
[train] Processing 1000/1512
[train] Processing 1200/1512
[train] Processing 1400/1512
[DONE] train: 1661 crops
[test] Processing 0/187
[DONE] test: 199 crops
[val] Processing 0/370
[val] Processing 200/370
[DONE] val: 416 crops

=== DATASET SUMMARY ===

TRAIN:
  wood_chisel            245
  anvil                  218
  blacksmith_hammer      154
  blacksmith_tongs       180
  carpentry_hammer       189
  hand_planer            241
  hand_saw               199
  pottery_wheel          235

VAL:
  wood_chisel            74
  anvil                  60
  blacksmith_hammer      43
  blacksmith_tongs       51
  carpentry_hammer       53
  hand_planer            0
  hand_saw               69
  pottery_wheel          66

TEST:


In [ ]:
"""
Problem 3: Data Augmentation as Inductive Bias
Final version: atomic + combined augmentations (ablation study),
per-class evaluation, training curves, robustness gap analysis.
Hypothesis: augmentations teach invariances → better robustness to corruptions.
Deliverable: Augmentation-performance matrix + failure case analysis.
"""

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import torchvision.transforms.functional as TF
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")
import seaborn as sns
from pathlib import Path
from PIL import Image, ImageFilter
import json
import random

# ── CONFIG ────────────────────────────────────────────────────────────────────
CROPS_ROOT  = Path("/content/drive/MyDrive/dlcv_phase_2/IndiCraft-crops")
OUT_DIR     = Path("/content/drive/MyDrive/dlcv_phase_2/results/unit1")
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS      = 10
BATCH       = 32
LR          = 1e-3
IMG_SIZE    = 224
NUM_CLASSES = 8
SEEDS       = [42, 43, 44]   # multi-seed for variance estimates

CLASS_NAMES = [
    "wood_chisel", "anvil", "blacksmith_hammer", "blacksmith_tongs",
    "carpentry_hammer", "hand_planer", "hand_saw", "pottery_wheel"
]

OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── COMMON TRANSFORMS ─────────────────────────────────────────────────────────
normalize = transforms.Normalize([0.485, 0.456, 0.406],
                                 [0.229, 0.224, 0.225])

base_eval = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    normalize,
])

# ── AUGMENTATION STRATEGIES ───────────────────────────────────────────────────
# Rule: ToTensor → normalize → RandomErasing (always last, always post-normalize)
AUGMENTATION_STRATEGIES = {

    # ── BASELINE ──────────────────────────────────────────────────────────────
    "Baseline": transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        normalize,
    ]),

    # ── ATOMIC (ablation) ─────────────────────────────────────────────────────
    "Rotation only": transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomRotation(30),
        transforms.ToTensor(),
        normalize,
    ]),

    "Crop only": transforms.Compose([
        transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
        transforms.RandomCrop(IMG_SIZE),
        transforms.ToTensor(),
        normalize,
    ]),

    "Flip only": transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        normalize,
    ]),

    "Color jitter only": transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ColorJitter(0.5, 0.5, 0.5, 0.1),
        transforms.ToTensor(),
        normalize,
    ]),

    "Cutout only": transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        normalize,                                           # ✅ before erasing
        transforms.RandomErasing(p=0.5, scale=(0.05, 0.25)),
    ]),

    # ── COMBINED ──────────────────────────────────────────────────────────────
    "Geometric combo": transforms.Compose([
        transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
        transforms.RandomCrop(IMG_SIZE),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(p=0.1),
        transforms.RandomRotation(30),
        transforms.ToTensor(),
        normalize,
    ]),

    "Color combo": transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ColorJitter(0.5, 0.5, 0.5, 0.1),
        transforms.ToTensor(),
        normalize,
        transforms.RandomErasing(p=0.5, scale=(0.05, 0.25)),
    ]),

    "Full mix": transforms.Compose([
        transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
        transforms.RandomCrop(IMG_SIZE),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(p=0.1),
        transforms.RandomRotation(30),
        transforms.ColorJitter(0.5, 0.5, 0.5, 0.1),
        transforms.ToTensor(),
        normalize,                                           # ✅ before erasing
        transforms.RandomErasing(p=0.5, scale=(0.05, 0.25)),
    ]),
}

# ── CORRUPTIONS ───────────────────────────────────────────────────────────────
class GaussianNoise:
    """Simulate sensor noise (std=25)."""
    def __call__(self, img: Image.Image) -> Image.Image:
        arr = np.array(img).astype(np.float32)
        noise = np.random.randn(*arr.shape) * 25
        return Image.fromarray(np.clip(arr + noise, 0, 255).astype(np.uint8))

class GaussianBlur:
    """Simulate camera defocus (radius=3)."""
    def __call__(self, img: Image.Image) -> Image.Image:
        return img.filter(ImageFilter.GaussianBlur(radius=3))

class BrightnessShift:
    """Simulate lighting variation."""
    def __call__(self, img: Image.Image) -> Image.Image:
        return TF.adjust_brightness(img, random.uniform(0.3, 1.7))

class LowContrast:
    """Simulate washed-out imagery."""
    def __call__(self, img: Image.Image) -> Image.Image:
        return TF.adjust_contrast(img, 0.3)

CORRUPTIONS = {
    "Clean":       None,
    "Gauss noise": GaussianNoise(),
    "Blur":        GaussianBlur(),
    "Brightness":  BrightnessShift(),
    "Low contrast": LowContrast(),
}

def make_corrupt_transform(corruption):
    steps = [transforms.Resize((IMG_SIZE, IMG_SIZE))]
    if corruption:
        steps.append(corruption)
    steps += [transforms.ToTensor(), normalize]
    return transforms.Compose(steps)

# ── MODEL ─────────────────────────────────────────────────────────────────────
def get_model() -> nn.Module:
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    return model.to(DEVICE)

# ── SEED ──────────────────────────────────────────────────────────────────────
def set_seed(seed: int):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)

# ── TRAIN ─────────────────────────────────────────────────────────────────────
def train_model(aug_name: str, train_transform, seed: int):
    """Train ResNet-18 and return (model, history)."""
    print(f"\n{'='*60}")
    print(f"Training : {aug_name}  |  seed={seed}  |  device={DEVICE}")
    print('='*60)
    set_seed(seed)

    train_ds = datasets.ImageFolder(str(CROPS_ROOT / "train"), transform=train_transform)
    val_ds   = datasets.ImageFolder(str(CROPS_ROOT / "val"),   transform=base_eval)

    train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                          num_workers=2, pin_memory=True)
    val_dl   = DataLoader(val_ds,   batch_size=BATCH,
                          num_workers=2, pin_memory=True)

    model     = get_model()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    history = {"train_loss": [], "val_acc": []}

    for epoch in range(1, EPOCHS + 1):
        # training
        model.train()
        running_loss = 0.0
        for imgs, labels in train_dl:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * imgs.size(0)
        scheduler.step()

        epoch_loss = running_loss / len(train_ds)
        history["train_loss"].append(epoch_loss)

        # validation
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for imgs, labels in val_dl:
                preds = model(imgs.to(DEVICE)).argmax(1).cpu()
                correct += (preds == labels).sum().item()
                total   += labels.size(0)

        val_acc = correct / total
        history["val_acc"].append(val_acc)
        print(f"  Epoch {epoch:>2}/{EPOCHS}  "
              f"train_loss={epoch_loss:.4f}  val_acc={val_acc:.3f}")

    return model, history

# ── EVALUATE (overall + per-class) ────────────────────────────────────────────
def evaluate(model: nn.Module, corruption) -> dict:
    """Return overall and per-class accuracy on the test split."""
    tfm = make_corrupt_transform(corruption)
    ds  = datasets.ImageFolder(str(CROPS_ROOT / "test"), transform=tfm)
    dl  = DataLoader(ds, batch_size=BATCH, num_workers=2, pin_memory=True)

    model.eval()
    class_correct = [0] * NUM_CLASSES
    class_total   = [0] * NUM_CLASSES

    with torch.no_grad():
        for imgs, labels in dl:
            preds = model(imgs.to(DEVICE)).argmax(1).cpu()
            for pred, label in zip(preds, labels):
                class_correct[label] += int(pred == label)
                class_total[label]   += 1

    per_class = [class_correct[i] / max(class_total[i], 1) for i in range(NUM_CLASSES)]
    overall   = sum(class_correct) / max(sum(class_total), 1)
    return {"overall": overall, "per_class": per_class}

# ── PLOTS ─────────────────────────────────────────────────────────────────────
def plot_training_curves(all_histories: dict):
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    for aug_name, runs in all_histories.items():
        mean_loss = np.mean([r["train_loss"] for r in runs], axis=0)
        mean_acc  = np.mean([r["val_acc"]    for r in runs], axis=0)
        epochs    = range(1, len(mean_loss) + 1)
        axes[0].plot(epochs, mean_loss, label=aug_name, marker="o", markersize=3)
        axes[1].plot(epochs, mean_acc,  label=aug_name, marker="o", markersize=3)

    for ax, title, ylabel in zip(
        axes,
        ["Training Loss per Epoch", "Validation Accuracy per Epoch"],
        ["Cross-Entropy Loss", "Accuracy"]
    ):
        ax.set_title(title)
        ax.set_xlabel("Epoch")
        ax.set_ylabel(ylabel)
        ax.legend(fontsize=7, ncol=2)
        ax.grid(alpha=0.3)

    plt.suptitle("Training Curves – All Augmentation Strategies", fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "training_curves.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("Saved: training_curves.png")


def plot_overall_heatmap(results: dict):
    augs  = list(results.keys())
    corrs = list(CORRUPTIONS.keys())
    matrix = np.array([[results[a][c]["overall"] for c in corrs] for a in augs])

    plt.figure(figsize=(12, 7))
    ax = sns.heatmap(
        matrix, annot=True, fmt=".3f", cmap="RdYlGn",
        vmin=0.0, vmax=1.0,
        xticklabels=corrs, yticklabels=augs,
        linewidths=0.5, linecolor="grey",
    )
    # draw separator line between atomic and combined strategies
    n_atomic = sum(1 for k in augs if "only" in k or k == "Baseline")
    ax.axhline(n_atomic, color="black", linewidth=2.5, linestyle="--")

    ax.set_title("Overall Accuracy: Augmentation × Corruption\n"
                 "(dashed line separates atomic from combined)", fontsize=12)
    ax.set_xlabel("Test Corruption", fontsize=11)
    ax.set_ylabel("Training Augmentation", fontsize=11)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "heatmap_overall.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("Saved: heatmap_overall.png")


def plot_perclass_heatmap(results: dict):
    augs   = list(results.keys())
    matrix = np.array([results[a]["Clean"]["per_class"] for a in augs])

    plt.figure(figsize=(14, 7))
    ax = sns.heatmap(
        matrix, annot=True, fmt=".2f", cmap="RdYlGn",
        vmin=0.0, vmax=1.0,
        xticklabels=CLASS_NAMES, yticklabels=augs,
        linewidths=0.5, linecolor="grey",
    )
    ax.set_title("Per-Class Accuracy on Clean Test Set\n(Failure Case Analysis)", fontsize=12)
    ax.set_xlabel("Class", fontsize=11)
    ax.set_ylabel("Training Augmentation", fontsize=11)
    plt.xticks(rotation=30, ha="right", fontsize=9)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "heatmap_perclass_clean.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("Saved: heatmap_perclass_clean.png")


def plot_robustness_gap(results: dict):
    augs  = list(results.keys())
    corrs = [c for c in CORRUPTIONS if c != "Clean"]
    x     = np.arange(len(corrs))
    width = 0.85 / len(augs)

    fig, ax = plt.subplots(figsize=(14, 5))
    for i, aug in enumerate(augs):
        clean = results[aug]["Clean"]["overall"]
        deltas = [clean - results[aug][c]["overall"] for c in corrs]
        ax.bar(x + i * width, deltas, width, label=aug)

    ax.set_title("Robustness Gap: Clean − Corrupted Accuracy (↓ is better)", fontsize=12)
    ax.set_xlabel("Corruption Type")
    ax.set_ylabel("Accuracy Drop")
    ax.set_xticks(x + width * (len(augs) - 1) / 2)
    ax.set_xticklabels(corrs)
    ax.legend(fontsize=7, ncol=3)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "robustness_gap.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("Saved: robustness_gap.png")


def plot_ablation_bar(results: dict):
    """Clean accuracy bar chart — atomic vs combined, easy to compare."""
    augs        = list(results.keys())
    clean_accs  = [results[a]["Clean"]["overall"] for a in augs]
    colors      = ["#4C72B0" if ("only" in a or a == "Baseline") else "#DD8452"
                   for a in augs]

    fig, ax = plt.subplots(figsize=(13, 5))
    bars = ax.bar(augs, clean_accs, color=colors, edgecolor="grey", linewidth=0.5)
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel("Clean Test Accuracy")
    ax.set_title("Ablation Study: Clean Accuracy per Augmentation Strategy\n"
                 "(Blue = atomic  |  Orange = combined)", fontsize=12)
    plt.xticks(rotation=30, ha="right", fontsize=9)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "ablation_clean_acc.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("Saved: ablation_clean_acc.png")


def print_failure_analysis(results: dict):
    lines = []
    sep = "=" * 70
    lines += [sep, "FAILURE CASE ANALYSIS", sep]

    for aug_name, corr_dict in results.items():
        lines.append(f"\n[ {aug_name} ]")
        clean_acc = corr_dict["Clean"]["overall"]
        lines.append(f"  Clean accuracy       : {clean_acc:.3f}")

        worst_c = min(
            (c for c in corr_dict if c != "Clean"),
            key=lambda c: corr_dict[c]["overall"]
        )
        lines.append(f"  Worst corruption     : {worst_c} "
                     f"({corr_dict[worst_c]['overall']:.3f})")

        pc = corr_dict["Clean"]["per_class"]
        worst_cls = int(np.argmin(pc))
        best_cls  = int(np.argmax(pc))
        lines.append(f"  Hardest class (clean): {CLASS_NAMES[worst_cls]} "
                     f"({pc[worst_cls]:.3f})")
        lines.append(f"  Easiest class (clean): {CLASS_NAMES[best_cls]} "
                     f"({pc[best_cls]:.3f})")

        avg_drop = float(np.mean([
            clean_acc - corr_dict[c]["overall"]
            for c in corr_dict if c != "Clean"
        ]))
        lines.append(f"  Avg robustness gap   : {avg_drop:.3f}")

    lines += [
        f"\n{sep}",
        "HYPOTHESIS EVALUATION",
        sep,
        "Augmentation teaches invariances:",
        "  - Geometric (crop/flip/rotation) → spatial invariance"
        " → should reduce Blur gap",
        "  - Color jitter → photometric invariance"
        " → should reduce Brightness / LowContrast gap",
        "  - Cutout → occlusion robustness"
        " → may help Noise but not Blur",
        "  - Full mix → should dominate all corruptions",
        "  - Atomic ablation isolates which invariance each aug provides",
        "  Compare atomic rows vs combined rows in heatmap_overall.png",
    ]

    report = "\n".join(lines)
    print(report)
    with open(OUT_DIR / "failure_analysis.txt", "w") as f:
        f.write(report)
    print("Saved: failure_analysis.txt")


# ── MAIN ──────────────────────────────────────────────────────────────────────
def main():
    all_results   = {}   # aug_name → {corruption → eval_dict}
    all_histories = {}   # aug_name → list[history_dict]

    for aug_name, tfm in AUGMENTATION_STRATEGIES.items():
        print(f"\n{'#'*60}")
        print(f"# STRATEGY: {aug_name}")
        print(f"{'#'*60}")

        seed_results   = []
        seed_histories = []

        for seed in SEEDS:
            model, history = train_model(aug_name, tfm, seed)
            seed_histories.append(history)

            corr_evals = {}
            print(f"\n  [Evaluation — seed={seed}]")
            for cname, cfn in CORRUPTIONS.items():
                eval_dict = evaluate(model, cfn)
                corr_evals[cname] = eval_dict
                print(f"    {cname:15s} overall={eval_dict['overall']:.3f}")
            seed_results.append(corr_evals)

        all_histories[aug_name] = seed_histories

        # average over seeds
        averaged = {}
        for cname in CORRUPTIONS:
            avg_overall   = float(np.mean([r[cname]["overall"] for r in seed_results]))
            avg_per_class = list(np.mean(
                [r[cname]["per_class"] for r in seed_results], axis=0
            ))
            averaged[cname] = {"overall": avg_overall, "per_class": avg_per_class}

        all_results[aug_name] = averaged

    # ── SAVE ──────────────────────────────────────────────────────────────────
    with open(OUT_DIR / "results.json", "w") as f:
        json.dump(all_results, f, indent=2)
    print("\nSaved: results.json")

    # ── PLOTS ─────────────────────────────────────────────────────────────────
    plot_training_curves(all_histories)
    plot_overall_heatmap(all_results)
    plot_perclass_heatmap(all_results)
    plot_robustness_gap(all_results)
    plot_ablation_bar(all_results)
    print_failure_analysis(all_results)

    print(f"\n✅  All outputs saved to: {OUT_DIR}")


if __name__ == "__main__":
    main()


############################################################
# STRATEGY: Baseline
############################################################

Training : Baseline  |  seed=42  |  device=cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 128MB/s]


  Epoch  1/10  train_loss=1.0337  val_acc=0.257
  Epoch  2/10  train_loss=0.5494  val_acc=0.341
  Epoch  3/10  train_loss=0.2651  val_acc=0.380
  Epoch  4/10  train_loss=0.1465  val_acc=0.346
  Epoch  5/10  train_loss=0.1264  val_acc=0.401
  Epoch  6/10  train_loss=0.0535  val_acc=0.421
  Epoch  7/10  train_loss=0.0216  val_acc=0.425
  Epoch  8/10  train_loss=0.0083  val_acc=0.430
  Epoch  9/10  train_loss=0.0059  val_acc=0.425
  Epoch 10/10  train_loss=0.0063  val_acc=0.425

  [Evaluation — seed=42]
    Clean           overall=0.482
    Gauss noise     overall=0.432
    Blur            overall=0.221
    Brightness      overall=0.467
    Low contrast    overall=0.437

Training : Baseline  |  seed=43  |  device=cuda
  Epoch  1/10  train_loss=0.9378  val_acc=0.305
  Epoch  2/10  train_loss=0.4465  val_acc=0.353
  Epoch  3/10  train_loss=0.2471  val_acc=0.358
  Epoch  4/10  train_loss=0.1447  val_acc=0.389
  Epoch  5/10  train_loss=0.1142  val_acc=0.416
  Epoch  6/10  train_loss=0.0334  v